# Final experiment: fair comparison, evidence analyses and the Soundtracks test

This is the final, pre-declared experiment of the thesis. It improves the weak points found in the Training Three Folds notebook and adds the analyses needed to *explain* the results with evidence:

| Part | What | Why |
|---|---|---|
| **A** | Three folds, 10 seeds: baseline, **stronger** domain alignment (4 strengths), emotion alignment (4 strengths, wider range); locked choices; all-pairs control; one test-domain evaluation | fair comparison; the Training Three Folds notebook's best emotion strength was at the edge of its range and the DANN discriminator was weak |
| **B** | **In-domain reference**: for each domain, a model trained and tested *inside* that domain | shows how much room for improvement the unseen-domain setting leaves (e.g. why MediaEval shows no gain) |
| **C** | **Probes and distances** (no test data): how much *emotion* and how much *domain* information each method keeps; cross-domain neighbour agreement; how far apart the domains are | evidence for the thesis argument that domain alignment can remove emotion information |
| **D** | **Final independent test on Soundtracks** (never used before): models trained on all three domains with strengths fixed by a rule declared below | the clean confirmation of the thesis claim |
| **E** | Tables, paired tests, bootstrap intervals, review file | reporting |

**Changes compared with the Training Three Folds notebook (decided before seeing any result of this notebook):**
* The feature mode is fixed to `mix` (all 13 layers), as locked in the Training Three Folds notebook from source validation.
* The layer weights get their own, larger learning rate (and no weight decay), so they can really be learned.
* Domain alignment: a larger discriminator (128→256→256) with a 10× learning rate, and the strengths 0.03, 0.1, 0.3, 1.0.
* Emotion alignment strengths 0.1, 0.2, 0.4, 0.8.
* 10 seeds; "better" requires winning in at least 8 of 10 seeds **and** a bootstrap interval above zero.

**Honesty notes:** the three folds were already evaluated once in the Training Three Folds notebook, so this run's fold results confirm a development finding; **Soundtracks** is the only never-seen test set. PMEmo is exploratory.

**Run:** GPU (L4) → Run all. Resumable: after a disconnect, Run all again.
**Send back:** `final_review.zip`.

## Step 1 — Settings (the protocol is declared here, before any result)

In [1]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/Thesis - Music Emotion Recognition')
DATA_DIR = DRIVE_PROJECT / 'MER_DG_v2'                 # outputs of the Data Preparation notebook
DATASETS_DIR = DRIVE_PROJECT / 'Datasets'              # raw datasets (only the Soundtracks ratings are read, in Part D)

FOLDS = {'MediaEval': ['PMEmo', 'Memo2496'], 'Memo2496': ['MediaEval', 'PMEmo'], 'PMEmo': ['MediaEval', 'Memo2496']}
ALL_DOMAINS = ['MediaEval', 'Memo2496', 'PMEmo']
SEEDS = list(range(1, 11))
REFERENCE_SEEDS = [1, 2, 3]                            # Part B
PROBE_SEEDS = [1, 2, 3, 4, 5]                          # Part C
VAL_FRACTION, INNER_VAL_FRACTION, SPLIT_SEED = 0.2, 0.15, 2026
NEAR_DUPLICATE_COSINE = 0.98
FEATURE_MODE = 'mix'                                   # locked in the Training Three Folds notebook
TRAIN = dict(epochs=30, batch_per_domain=32, learning_rate=5e-4, weight_decay=1e-3, warmup_epochs=1,
             grad_clip=5.0, dropout=0.3, hidden=256, latent=128,
             layer_lr_factor=100.0,                    # layer weights learn 100x faster, no weight decay
             disc_lr_factor=10.0, disc_hidden=256)     # stronger DANN discriminator
DANN_WEIGHTS = [0.03, 0.1, 0.3, 1.0]
EMOTION_WEIGHTS = [0.1, 0.2, 0.4, 0.8]
EMOTION = dict(temperature=0.1, bandwidth=0.15)
BETTER_RULE = dict(min_seed_wins=8, ci=0.95)
BOOTSTRAP_REPS = 2000
FINAL_WEIGHT_RULE = ('For Soundtracks, each method uses the strength chosen most often across the three '
                     'folds (source-validation choices); if all three differ, the median.')

PROTOCOL = dict(version='nb18-final-v1', folds=FOLDS, seeds=SEEDS, reference_seeds=REFERENCE_SEEDS,
                probe_seeds=PROBE_SEEDS, val_fraction=VAL_FRACTION, inner_val_fraction=INNER_VAL_FRACTION,
                split_seed=SPLIT_SEED, near_duplicate_cosine=NEAR_DUPLICATE_COSINE, feature_mode=FEATURE_MODE,
                train=TRAIN, dann_weights=DANN_WEIGHTS, emotion_weights=EMOTION_WEIGHTS, emotion=EMOTION,
                better_rule=BETTER_RULE, bootstrap_reps=BOOTSTRAP_REPS, final_weight_rule=FINAL_WEIGHT_RULE,
                selection='best mean source-validation macro CCC over seeds; ties -> weaker strength',
                memo_window='15-75')
print('Protocol declared.')

Protocol declared.


## Step 2 — Connect Drive, check inputs, create the experiment folder

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, sys
import torch

def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

for needed in ['catalog/catalog.csv', 'catalog/soundtracks_lock.json', 'features/manifest.csv',
               'features/near_duplicates.csv']:
    if not (DATA_DIR / needed).is_file():
        raise FileNotFoundError(f'Missing {DATA_DIR / needed}. Run Notebook 16 first or fix DATA_DIR.')
INPUTS = {n: sha256_file(DATA_DIR / n) for n in ['catalog/catalog.csv', 'features/manifest.csv',
                                                  'features/near_duplicates.csv', 'catalog/soundtracks_lock.json']}
EXPERIMENT_ID = sha256_json(dict(protocol=PROTOCOL, inputs=INPUTS))[:12]
RUN_DIR = DATA_DIR / 'experiments' / f'nb18_{EXPERIMENT_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'protocol.json').write_text(json.dumps(dict(protocol=PROTOCOL, inputs=INPUTS), indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → GPU (L4 recommended), then Run all.')
DEVICE = torch.device('cuda')
print(f'✅ Experiment folder: {RUN_DIR}\n   GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}')

Mounted at /content/drive
✅ Experiment folder: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/experiments/nb18_97de095f25de
   GPU: NVIDIA L4 | PyTorch 2.11.0+cu128


## Step 3 — Helpers and metrics

CCC (primary, higher is better), Pearson r, MAE and RMSE (0–1 scale; ×100 = points). A progress reporter prints the remaining time.

In [3]:
import datetime, math, time, threading, zipfile, zlib, io, csv
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

def log(message):
    print(f'[{datetime.datetime.now():%H:%M:%S}] {message}', flush=True)

def fmt_seconds(s):
    if s is None or not math.isfinite(s):
        return '?'
    s = int(s)
    return f'{s // 3600}h {s % 3600 // 60:02d}m' if s >= 3600 else f'{s // 60}m {s % 60:02d}s'

class Progress:
    def __init__(self, total, label, every_s=30):
        self.total, self.label, self.every_s = max(int(total), 1), label, every_s
        self.done, self.start, self.last, self.lock = 0, time.monotonic(), -1e9, threading.Lock()
        log(f'▶ {label}: {total} to do')
    def step(self, n=1, note=''):
        with self.lock:
            self.done += n
            now = time.monotonic()
            if self.done >= self.total or now - self.last >= self.every_s:
                elapsed = now - self.start
                rate = self.done / elapsed if elapsed > 0 else 0
                eta = (self.total - self.done) / rate if rate > 0 else None
                log(f'  {self.label}: {self.done}/{self.total} ({100 * self.done / self.total:.0f}%) | '
                    f'elapsed {fmt_seconds(elapsed)} | remaining ≈ {fmt_seconds(eta)} {note}')
                self.last = now

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False,
                              default=lambda v: v.item() if hasattr(v, 'item') else str(v)))
    os.replace(tmp, path)

AXES = ('valence', 'arousal')

def axis_metrics(truth, pred):
    truth, pred = np.asarray(truth, np.float64), np.asarray(pred, np.float64)
    mt, mp, vt, vp = truth.mean(), pred.mean(), truth.var(), pred.var()
    cov = ((truth - mt) * (pred - mp)).mean()
    err = pred - truth
    return dict(ccc=float(2 * cov / (vt + vp + (mt - mp) ** 2)) if vt + vp > 0 else float('nan'),
                pearson=float(cov / math.sqrt(vt * vp)) if vt > 0 and vp > 0 else float('nan'),
                mae=float(np.abs(err).mean()), rmse=float(np.sqrt((err ** 2).mean())), bias=float(err.mean()))

def metrics(y, pred):
    out = {}
    for i, axis in enumerate(AXES):
        for k, v in axis_metrics(y[:, i], pred[:, i]).items():
            out[f'{axis}_{k}'] = v
    for k in ('ccc', 'pearson', 'mae', 'rmse'):
        out[f'macro_{k}'] = (out[f'valence_{k}'] + out[f'arousal_{k}']) / 2
    return out

def ccc_batch(truth, pred):
    mt, mp = truth.mean(-1, keepdims=True), pred.mean(-1, keepdims=True)
    cov = ((truth - mt) * (pred - mp)).mean(-1)
    return 2 * cov / (truth.var(-1) + pred.var(-1) + (mt - mp)[..., 0] ** 2)

_y = np.array([[0.1, 0.2], [0.5, 0.6], [0.9, 0.7]])
assert abs(metrics(_y, _y)['macro_ccc'] - 1) < 1e-12
log('Helpers ready.')

[15:00:21] Helpers ready.


## Step 4 — Catalog, labels and splits

Same rules as the Training Three Folds notebook (artist groups + near-duplicate pairs never split; 20 % source validation per domain, seed 2026), so the splits are identical to the Training Three Folds notebook.
Additionally, for Part B, 15 % of each domain's *training* groups form an inner validation set.
Soundtracks is kept out of all training data; its features and ratings are loaded only in Part D.

In [4]:
catalog_all = pd.read_csv(DATA_DIR / 'catalog' / 'catalog.csv', dtype={'track_id': str}, keep_default_na=False)
catalog = catalog_all[catalog_all['dataset'] != 'Soundtracks'].reset_index(drop=True)
catalog['valence'] = catalog['valence'].astype(float)
catalog['arousal'] = catalog['arousal'].astype(float)

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)
for uid, group in zip(catalog['uid'], catalog['group_id']):
    union(uid, 'group:' + group)
near = pd.read_csv(DATA_DIR / 'features' / 'near_duplicates.csv')
ids = set(catalog['uid'])
near = near[(near['cosine'] >= NEAR_DUPLICATE_COSINE) & near['uid_a'].isin(ids) & near['uid_b'].isin(ids)]
for a, b in zip(near['uid_a'], near['uid_b']):
    union(a, b)
catalog['component'] = [find(u) for u in catalog['uid']]

def pick_components(frame, fraction, seed_parts):
    sizes = frame.groupby('component').size()
    order = np.random.default_rng(seed_parts).permutation(sorted(sizes.index))
    chosen, count = [], 0
    for comp in order:
        if count >= fraction * len(frame):
            break
        chosen.append(comp)
        count += sizes[comp]
    return set(chosen)

catalog['split'] = 'train'
for domain, part in catalog.groupby('domain'):
    val = pick_components(part, VAL_FRACTION, [SPLIT_SEED, zlib.crc32(domain.encode())])
    catalog.loc[(catalog.domain == domain) & catalog.component.isin(val), 'split'] = 'val'
catalog['inner'] = 'fit'
for domain, part in catalog[catalog.split == 'train'].groupby('domain'):
    inner = pick_components(part, INNER_VAL_FRACTION, [SPLIT_SEED + 1, zlib.crc32(domain.encode())])
    catalog.loc[(catalog.domain == domain) & (catalog.split == 'train') & catalog.component.isin(inner), 'inner'] = 'inner_val'

display(catalog.groupby(['domain', 'split']).size().unstack(fill_value=0))
catalog[['uid', 'domain', 'split', 'inner', 'component']].to_csv(RUN_DIR / 'splits.csv', index=False)
log(f'✅ {len(catalog)} songs (Soundtracks excluded), splits saved.')

def tracks(domain, split=None, inner=None):
    mask = catalog.domain == domain
    if split:
        mask &= catalog.split == split
    if inner:
        mask &= catalog.inner == inner
    return np.flatnonzero(mask.to_numpy())

split,train,val
domain,,
MediaEval,1430,358
Memo2496,1931,483
PMEmo,603,151


[15:00:21] ✅ 4956 songs (Soundtracks excluded), splits saved.


## Step 5 — Load the features into GPU memory

Reuses the feature pack made by the Training Three Folds notebook (built here if missing). Memo2496 uses only windows starting in 15–75 s for tracks ≥ 30 s, as its labels.

In [5]:
manifest_all = pd.read_csv(DATA_DIR / 'features' / 'manifest.csv')
manifest = manifest_all[manifest_all['dataset'] != 'Soundtracks'].reset_index(drop=True)
pack_path = DATA_DIR / 'features' / 'pack_no_soundtracks.npz'
pack_meta = dict(manifest_sha256=INPUTS['features/manifest.csv'])

def build_pack():
    progress = Progress(len(manifest), 'Reading feature files from Drive (one time only)')
    results = [None] * len(manifest)
    def task(i):
        with np.load(DATA_DIR / manifest.loc[i, 'feature_relpath'], allow_pickle=False) as f:
            results[i] = (f['layers'].astype(np.float16), f['start_s'].astype(np.float32))
        progress.step()
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(task, range(len(manifest))))
    tmp = pack_path.with_name(pack_path.name + '.tmp.npz')
    np.savez(tmp, layers=np.concatenate([r[0] for r in results]), starts=np.concatenate([r[1] for r in results]),
             counts=np.array([len(r[1]) for r in results]), uids=manifest['uid'].to_numpy(dtype=str),
             meta=np.array(json.dumps(pack_meta)))
    os.replace(tmp, pack_path)

need = True
if pack_path.is_file():
    with np.load(pack_path, allow_pickle=False) as f:
        need = json.loads(str(f['meta'])) != pack_meta
if need:
    build_pack()
log('Loading the feature pack ...')
with np.load(pack_path, allow_pickle=False) as f:
    PACK_LAYERS = torch.from_numpy(f['layers']).to(DEVICE)
    pack_starts, pack_counts, pack_uids = f['starts'], f['counts'], list(f['uids'])
N_LAYERS, WIDTH = PACK_LAYERS.shape[1], PACK_LAYERS.shape[2]
offsets = np.concatenate([[0], np.cumsum(pack_counts)[:-1]])
pack_pos = {u: i for i, u in enumerate(pack_uids)}

def window_index(uid, dataset, duration):
    i = pack_pos[uid]
    idx = np.arange(offsets[i], offsets[i] + pack_counts[i])
    if dataset == 'Memo2496' and float(duration) >= 30:
        s = pack_starts[idx]
        keep = (s >= 15 - 1e-3) & (s < 75 - 1e-3)
        idx = idx[keep] if keep.any() else idx
    return idx

lists = [window_index(r.uid, r.dataset, r.duration_s) for r in catalog.itertuples()]
WIN = torch.full((len(catalog), max(len(w) for w in lists)), -1, dtype=torch.long)
for t, idx in enumerate(lists):
    WIN[t, :len(idx)] = torch.from_numpy(idx)
WIN = WIN.to(DEVICE)
LENGTHS = torch.tensor([len(w) for w in lists], device=DEVICE)
Y = torch.tensor(catalog[['valence', 'arousal']].to_numpy(), dtype=torch.float32, device=DEVICE)
log(f'✅ {PACK_LAYERS.shape[0]} windows on GPU ({PACK_LAYERS.numel() * 2 / 2**30:.2f} GB).')

[15:00:25] Loading the feature pack ...
[15:00:50] ✅ 100561 windows on GPU (1.87 GB).


## Step 6 — Model and losses (with self-tests)

Same architecture as the Training Three Folds notebook, with two fairness fixes: layer weights learn with their own larger learning rate, and the DANN discriminator is larger and learns 10× faster. A self-test checks that padded windows have no effect, the gradient reversal sign, and finite losses.

In [6]:
from torch import nn
from torch.nn import functional as F

class SongModel(nn.Module):
    def __init__(self, n_layers, width, hidden, latent, dropout):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.window = nn.Sequential(nn.Linear(width, hidden), nn.GELU(), nn.Dropout(dropout))
        self.attention = nn.Linear(hidden, 1)
        self.project = nn.Sequential(nn.Linear(2 * hidden, latent), nn.LayerNorm(latent), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Linear(latent, 2)
    def layer_weights(self):
        return torch.softmax(self.layer_logits, 0)
    def embed(self, x, mask):
        x = (x * self.layer_weights()[None, None, :, None]).sum(2)
        h = self.window(x)
        a = self.attention(h).squeeze(-1).masked_fill(~mask, float('-inf')).softmax(1)[..., None]
        mean = (a * h).sum(1)
        std = ((a * (h - mean[:, None]) ** 2).sum(1) + 1e-5).sqrt()
        return self.project(torch.cat([mean, std], 1))
    def forward(self, x, mask):
        z = self.embed(x, mask)
        return self.head(z), z

class _Reverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, c):
        ctx.c = c
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.c * g, None

def gradient_reverse(x, c):
    return _Reverse.apply(x, c)

def emotion_loss(z, y, domains, scope, temperature, bandwidth):
    eye = torch.eye(len(y), dtype=torch.bool, device=z.device)
    mask = (domains[:, None] != domains[None, :]) if scope == 'cross' else ~eye
    valid = mask.any(1)
    unit = F.normalize(z, dim=1, eps=1e-8)
    mask = mask[valid]
    logits = (unit[valid] @ unit.T / temperature).masked_fill(~mask, float('-inf'))
    log_target = F.log_softmax((-torch.cdist(y[valid], y).square() / (2 * bandwidth ** 2)).masked_fill(~mask, float('-inf')), 1)
    return -(log_target.exp() * F.log_softmax(logits, 1).masked_fill(~mask, 0.)).sum(1).mean()

def ramp(p):
    return 2 / (1 + math.exp(-10 * p)) - 1

torch.manual_seed(0)
_z = torch.randn(9, 4, requires_grad=True)
gradient_reverse(_z, 0.5).sum().backward()
assert torch.allclose(_z.grad, torch.full_like(_z, -0.5))
_y, _d = torch.rand(9, 2), torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])
for scope in ('cross', 'all'):
    _z.grad = None
    l = emotion_loss(_z, _y, _d, scope, 0.1, 0.15)
    l.backward()
    assert torch.isfinite(l) and torch.isfinite(_z.grad).all()
_m = SongModel(13, 16, 8, 4, 0.0)
_x, _mask = torch.randn(3, 5, 13, 16), torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1], [1, 0, 0, 0, 0]], dtype=torch.bool)
_x2 = _x.clone(); _x2[0, 3:] = 1e3
assert torch.allclose(_m(_x, _mask)[0], _m(_x2, _mask)[0], atol=1e-5)
log('✅ Model and loss self-tests passed.')

[15:00:51] ✅ Model and loss self-tests passed.


## Step 7 — Training, prediction and embedding functions

A *run spec* says which songs to train on (one entry per domain), which songs choose the best epoch, the method, strength and seed. Batches contain the same number of songs from each training domain. Every finished run is saved to Drive and reused.

In [7]:
def song_batch(t, mean, std):
    t = torch.as_tensor(t, device=DEVICE)
    idx = WIN[t, :int(LENGTHS[t].max())]
    mask = idx >= 0
    x = (PACK_LAYERS[idx.clamp(min=0)].float() - mean) / std
    return x * mask[..., None, None], mask

STATS = {}
@torch.no_grad()
def statistics_for(key, train_tracks):
    if key not in STATS:
        s1 = torch.zeros(N_LAYERS, WIDTH, device=DEVICE, dtype=torch.float64)
        s2 = torch.zeros_like(s1)
        for k in range(0, len(train_tracks), 64):
            t = torch.as_tensor(train_tracks[k:k + 64], device=DEVICE)
            idx = WIN[t, :int(LENGTHS[t].max())]
            m = (idx >= 0).double()
            x = PACK_LAYERS[idx.clamp(min=0)].double()
            w = (m / m.sum(1, keepdim=True))[..., None, None]
            s1 += (x * w).sum((0, 1))
            s2 += (x ** 2 * w).sum((0, 1))
        mean = s1 / len(train_tracks)
        STATS[key] = (mean.float(), (s2 / len(train_tracks) - mean ** 2).clamp(min=1e-8).sqrt().float().clamp(min=1e-3))
    return STATS[key]

@torch.no_grad()
def predict(model, t, mean, std, layers=None, starts=None):
    model.eval()
    out = []
    for k in range(0, len(t), 64):
        x, mask = song_batch(t[k:k + 64], mean, std)
        out.append(model(x, mask)[0].clamp(0, 1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def embed(model, t, mean, std):
    model.eval()
    out = []
    for k in range(0, len(t), 64):
        x, mask = song_batch(t[k:k + 64], mean, std)
        out.append(model.embed(x, mask).cpu().numpy())
    return np.concatenate(out)

def val_score(model, val, mean, std):
    per = {d: metrics(catalog.loc[t, ['valence', 'arousal']].to_numpy(), predict(model, t, mean, std)) for d, t in val.items()}
    return float(np.mean([m['macro_ccc'] for m in per.values()])), per

def train_run(spec):
    """spec: name, stats_key, train {domain: tracks}, val {domain: tracks}, method, weight, scope, seed."""
    folder = RUN_DIR / 'runs' / spec['name']
    if (folder / 'result.json').is_file():
        return json.loads((folder / 'result.json').read_text())
    started = time.monotonic()
    train, val, seed, method, weight = spec['train'], spec['val'], spec['seed'], spec['method'], spec['weight']
    domains_list = list(train)
    mean, std = statistics_for(spec['stats_key'], np.concatenate(list(train.values())))
    torch.manual_seed(seed)
    model = SongModel(N_LAYERS, WIDTH, TRAIN['hidden'], TRAIN['latent'], TRAIN['dropout']).to(DEVICE)
    lr = TRAIN['learning_rate']
    groups = [dict(params=[p for n, p in model.named_parameters() if n != 'layer_logits'], lr=lr, base=lr),
              dict(params=[model.layer_logits], lr=lr * TRAIN['layer_lr_factor'], base=lr * TRAIN['layer_lr_factor'], weight_decay=0.0)]
    discriminator = None
    if method == 'dann':
        with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
            torch.manual_seed(seed + 9999)
            h = TRAIN['disc_hidden']
            discriminator = nn.Sequential(nn.Linear(TRAIN['latent'], h), nn.GELU(), nn.Linear(h, h), nn.GELU(),
                                          nn.Linear(h, len(domains_list))).to(DEVICE)
        groups.append(dict(params=list(discriminator.parameters()), lr=lr * TRAIN['disc_lr_factor'],
                           base=lr * TRAIN['disc_lr_factor']))
    optimizer = torch.optim.AdamW(groups, lr=lr, weight_decay=TRAIN['weight_decay'])
    params = [p for g in groups for p in g['params']]
    per = TRAIN['batch_per_domain'] * (2 if len(domains_list) == 1 else 1)
    steps = math.ceil(max(len(t) for t in train.values()) / per)
    total, warmup = steps * TRAIN['epochs'], steps * TRAIN['warmup_epochs']
    history, best, step = [], dict(score=-float('inf')), 0
    for epoch in range(TRAIN['epochs']):
        model.train()
        if discriminator is not None:
            discriminator.train()
        # Same batch order for every method of the same setting and seed (fair comparison).
        rng = np.random.default_rng([seed, epoch, zlib.crc32(spec['stats_key'].encode())])
        streams = [np.concatenate([rng.permutation(train[d]) for _ in range(math.ceil(steps * per / len(train[d])))])[:steps * per]
                   for d in domains_list]
        sums = dict(sup=0., aux=0., correct=0, total=0)
        for s in range(steps):
            factor = step / max(warmup, 1) if step < warmup else 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(total - warmup, 1)))
            for g in optimizer.param_groups:
                g['lr'] = g['base'] * factor
            t = np.concatenate([st[s * per:(s + 1) * per] for st in streams])
            dom = torch.arange(len(domains_list), device=DEVICE).repeat_interleave(per)
            x, mask = song_batch(t, mean, std)
            y = Y[torch.as_tensor(t, device=DEVICE)]
            pred, z = model(x, mask)
            sup = F.mse_loss(pred, y)
            strength = weight * ramp(step / total)
            if method == 'dann':
                logits = discriminator(gradient_reverse(z, strength))
                aux = F.cross_entropy(logits, dom)
                loss = sup + aux
                sums['correct'] += int((logits.argmax(1) == dom).sum())
                sums['total'] += len(dom)
            elif method == 'emotion':
                aux = emotion_loss(z, y, dom, spec['scope'], EMOTION['temperature'], EMOTION['bandwidth'])
                loss = sup + strength * aux
            else:
                aux, loss = sup.new_zeros(()), sup
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss in {spec["name"]}')
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, TRAIN['grad_clip'])
            optimizer.step()
            sums['sup'] += float(sup)
            sums['aux'] += float(aux)
            step += 1
        score, per_domain = val_score(model, val, mean, std)
        history.append(dict(epoch=epoch + 1, train_mse=sums['sup'] / steps, auxiliary=sums['aux'] / steps, strength=strength,
                            val_macro_ccc=score, disc_accuracy=sums['correct'] / sums['total'] if sums['total'] else None))
        if score > best['score']:
            best = dict(score=score, epoch=epoch + 1, per_domain=per_domain,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    folder.mkdir(parents=True, exist_ok=True)
    torch.save(best['state'], folder / 'best.pt')
    model.load_state_dict(best['state'])
    result = {k: spec[k] for k in ('name', 'stats_key', 'method', 'weight', 'scope', 'seed')}
    result.update(train_domains=domains_list, best_epoch=best['epoch'], val_macro_ccc=best['score'],
                  val_per_domain=best['per_domain'], seconds=time.monotonic() - started,
                  layer_weights=model.layer_weights().detach().cpu().tolist(),
                  final_disc_accuracy=history[-1]['disc_accuracy'])
    atomic_json(folder / 'history.json', history)
    atomic_json(folder / 'result.json', result)
    return result

def load_model(result):
    model = SongModel(N_LAYERS, WIDTH, TRAIN['hidden'], TRAIN['latent'], TRAIN['dropout']).to(DEVICE)
    model.load_state_dict(torch.load(RUN_DIR / 'runs' / result['name'] / 'best.pt', map_location=DEVICE))
    return model

def run_all(specs, label):
    results, progress = [], Progress(len(specs), label, every_s=0)
    for spec in specs:
        r = train_run(spec)
        results.append(r)
        progress.step(note=f'| {spec["name"]} → epoch {r["best_epoch"]}, source-val CCC {r["val_macro_ccc"]:.3f} ({r["seconds"]:.0f}s)')
    return results

def stats_of(key):
    """Normalization statistics of a setting, recomputed if this session has not trained it yet."""
    if key.startswith('fold_'):
        train_tracks = np.concatenate([tracks(d, 'train') for d in FOLDS[key[5:]]])
    elif key.startswith('indomain_'):
        train_tracks = tracks(key[9:], 'train', 'fit')
    elif key == 'final_all3':
        train_tracks = np.concatenate([tracks(d, 'train') for d in ALL_DOMAINS])
    else:
        raise KeyError(key)
    return statistics_for(key, train_tracks)

def fold_spec(test, method, weight, scope, seed):
    src = FOLDS[test]
    return dict(name=f'fold_{test}/{method}_w{weight}_{scope}/seed{seed}', stats_key=f'fold_{test}',
                train={d: tracks(d, 'train') for d in src}, val={d: tracks(d, 'val') for d in src},
                method=method, weight=weight, scope=scope, seed=seed)
log('✅ Training functions ready.')

[15:00:51] ✅ Training functions ready.


# Part A — Three folds: train, lock choices, control, test once

In [8]:
specs = []
for t in FOLDS:
    for seed in SEEDS:
        specs.append(fold_spec(t, 'baseline', 0.0, 'none', seed))
        specs += [fold_spec(t, 'dann', w, 'none', seed) for w in DANN_WEIGHTS]
        specs += [fold_spec(t, 'emotion', w, 'cross', seed) for w in EMOTION_WEIGHTS]
partA = pd.DataFrame(run_all(specs, 'Part A training runs'))
partA['test_domain'] = partA['stats_key'].str.replace('fold_', '', regex=False)

lock_path = RUN_DIR / 'selection_lock.json'
if lock_path.is_file():
    SELECTION = json.loads(lock_path.read_text())['selection']
else:
    SELECTION = {}
    for t in FOLDS:
        SELECTION[t] = {'baseline': 0.0}
        for m in ('dann', 'emotion'):
            means = partA[(partA.test_domain == t) & (partA.method == m)].groupby('weight')['val_macro_ccc'].mean()
            SELECTION[t][m] = float(means[means == means.max()].index.min())
    atomic_json(lock_path, dict(selection=SELECTION, locked=datetime.datetime.now().isoformat(),
                                note='Written before any test-domain prediction in this notebook.'))
display(pd.DataFrame(SELECTION).T)
for t in FOLDS:
    print(f'\nSource-validation CCC by strength (test domain {t}):')
    print(partA[partA.test_domain == t].groupby(['method', 'weight'])[['val_macro_ccc', 'final_disc_accuracy']].mean().round(4).to_string())

control = pd.DataFrame(run_all([fold_spec(t, 'emotion', SELECTION[t]['emotion'], 'all', s) for t in FOLDS for s in SEEDS],
                               'Part A control runs (emotion, all pairs)'))

def fold_models(t):
    return [('baseline', 'baseline', 0.0, 'none'), ('domain alignment', 'dann', SELECTION[t]['dann'], 'none'),
            ('emotion alignment', 'emotion', SELECTION[t]['emotion'], 'cross'),
            ('control: emotion all-pairs', 'emotion', SELECTION[t]['emotion'], 'all')]

rows, pred_rows = [], []
progress = Progress(len(FOLDS) * 4 * len(SEEDS), 'Test-domain evaluation (3 folds)', every_s=20)
for t in FOLDS:
    test = np.flatnonzero((catalog.domain == t).to_numpy())
    mean, std = stats_of(f'fold_{t}')
    truth = catalog.loc[test, ['valence', 'arousal']].to_numpy()
    for label, m, w, scope in fold_models(t):
        for seed in SEEDS:
            r = json.loads((RUN_DIR / 'runs' / fold_spec(t, m, w, scope, seed)['name'] / 'result.json').read_text())
            pred = predict(load_model(r), test, mean, std)
            rows.append(dict(test_domain=t, model=label, weight=w, seed=seed, **metrics(truth, pred)))
            pred_rows.append(pd.DataFrame(dict(test_domain=t, model=label, seed=seed, uid=catalog.loc[test, 'uid'].values,
                                               split=catalog.loc[test, 'split'].values,
                                               true_valence=truth[:, 0], pred_valence=pred[:, 0],
                                               true_arousal=truth[:, 1], pred_arousal=pred[:, 1])))
            progress.step()
fold_results = pd.DataFrame(rows)
fold_predictions = pd.concat(pred_rows)
fold_results.to_csv(RUN_DIR / 'fold_test_results_per_seed.csv', index=False)
fold_predictions.to_csv(RUN_DIR / 'fold_test_predictions.csv.gz', index=False, compression='gzip')
log('✅ Part A finished.')

[15:00:52] ▶ Part A training runs: 270 to do


/tmp/ipykernel_265/3150824702.py:114: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  sums['sup'] += float(sup)


[15:01:13]   Part A training runs: 1/270 (0%) | elapsed 0m 21s | remaining ≈ 1h 34m | fold_MediaEval/baseline_w0.0_none/seed1 → epoch 8, source-val CCC 0.764 (21s)
[15:01:25]   Part A training runs: 2/270 (1%) | elapsed 0m 33s | remaining ≈ 1h 15m | fold_MediaEval/dann_w0.03_none/seed1 → epoch 12, source-val CCC 0.762 (13s)
[15:01:38]   Part A training runs: 3/270 (1%) | elapsed 0m 45s | remaining ≈ 1h 08m | fold_MediaEval/dann_w0.1_none/seed1 → epoch 14, source-val CCC 0.748 (12s)
[15:01:50]   Part A training runs: 4/270 (1%) | elapsed 0m 58s | remaining ≈ 1h 04m | fold_MediaEval/dann_w0.3_none/seed1 → epoch 20, source-val CCC 0.718 (12s)
[15:02:02]   Part A training runs: 5/270 (2%) | elapsed 1m 10s | remaining ≈ 1h 02m | fold_MediaEval/dann_w1.0_none/seed1 → epoch 1, source-val CCC 0.659 (12s)
[15:02:16]   Part A training runs: 6/270 (2%) | elapsed 1m 24s | remaining ≈ 1h 01m | fold_MediaEval/emotion_w0.1_cross/seed1 → epoch 8, source-val CCC 0.776 (14s)
[15:02:30]   Part A training

,baseline,dann,emotion
MediaEval,0.0,0.03,0.2
Memo2496,0.0,0.03,0.4
PMEmo,0.0,0.03,0.2



Source-validation CCC by strength (test domain MediaEval):
                 val_macro_ccc  final_disc_accuracy
method   weight                                    
baseline 0.00           0.7667                  NaN
dann     0.03           0.7626               0.6038
         0.10           0.7478               0.5604
         0.30           0.7175               0.5388
         1.00           0.6551               0.5144
emotion  0.10           0.7778                  NaN
         0.20           0.7788                  NaN
         0.40           0.7777                  NaN
         0.80           0.7783                  NaN

Source-validation CCC by strength (test domain Memo2496):
                 val_macro_ccc  final_disc_accuracy
method   weight                                    
baseline 0.00           0.8132                  NaN
dann     0.03           0.8091               0.6318
         0.10           0.8006               0.5768
         0.30           0.7750               0.53

# Part B — In-domain reference ("how good could it be inside the domain?")

For each domain, a baseline is trained on that domain's own training songs (best epoch chosen on its inner validation songs) and tested on its source-validation songs. The unseen-domain models of Part A are evaluated on **the same songs**, so the numbers are directly comparable:

* **gap** = in-domain CCC − unseen-domain baseline CCC (how much is lost by not seeing the domain);
* **gap closed by a method** = (method − baseline) / gap.

A small gap means little room for any domain-generalization method. This is an analysis, not a selection: nothing here changes any choice.

In [9]:
specs = [dict(name=f'indomain_{d}/baseline/seed{s}', stats_key=f'indomain_{d}', train={d: tracks(d, 'train', 'fit')},
              val={d: tracks(d, 'train', 'inner_val')}, method='baseline', weight=0.0, scope='none', seed=s)
         for d in ALL_DOMAINS for s in REFERENCE_SEEDS]
reference = pd.DataFrame(run_all(specs, 'Part B in-domain reference runs'))
ref_rows = []
for d in ALL_DOMAINS:
    eval_t = tracks(d, 'val')
    truth = catalog.loc[eval_t, ['valence', 'arousal']].to_numpy()
    mean, std = stats_of(f'indomain_{d}')
    ref = np.mean([metrics(truth, predict(load_model(r), eval_t, mean, std))['macro_ccc']
                   for r in reference[reference.stats_key == f'indomain_{d}'].to_dict('records')])
    sub = fold_predictions[(fold_predictions.test_domain == d) & (fold_predictions.split == 'val')]
    row = dict(domain=d, n_songs=len(eval_t), in_domain_ccc=ref)
    for label in ('baseline', 'domain alignment', 'emotion alignment'):
        vals = [metrics(g[['true_valence', 'true_arousal']].to_numpy(), g[['pred_valence', 'pred_arousal']].to_numpy())['macro_ccc']
                for _, g in sub[sub.model == label].groupby('seed')]
        row[f'unseen_{label}_ccc'] = float(np.mean(vals))
    row['gap'] = row['in_domain_ccc'] - row['unseen_baseline_ccc']
    for label in ('domain alignment', 'emotion alignment'):
        row[f'gap_closed_by_{label}'] = (row[f'unseen_{label}_ccc'] - row['unseen_baseline_ccc']) / row['gap'] if row['gap'] > 0 else float('nan')
    ref_rows.append(row)
reference_table = pd.DataFrame(ref_rows)
reference_table.to_csv(RUN_DIR / 'in_domain_reference.csv', index=False)
display(reference_table.round(4))
log('✅ Part B finished.')

[16:20:43] ▶ Part B in-domain reference runs: 9 to do
[16:20:50]   Part B in-domain reference runs: 1/9 (11%) | elapsed 0m 06s | remaining ≈ 0m 55s | indomain_MediaEval/baseline/seed1 → epoch 12, source-val CCC 0.760 (7s)
[16:20:57]   Part B in-domain reference runs: 2/9 (22%) | elapsed 0m 13s | remaining ≈ 0m 48s | indomain_MediaEval/baseline/seed2 → epoch 20, source-val CCC 0.765 (7s)
[16:21:04]   Part B in-domain reference runs: 3/9 (33%) | elapsed 0m 20s | remaining ≈ 0m 41s | indomain_MediaEval/baseline/seed3 → epoch 14, source-val CCC 0.763 (7s)
[16:21:08]   Part B in-domain reference runs: 4/9 (44%) | elapsed 0m 24s | remaining ≈ 0m 30s | indomain_Memo2496/baseline/seed1 → epoch 19, source-val CCC 0.731 (4s)
[16:21:11]   Part B in-domain reference runs: 5/9 (56%) | elapsed 0m 28s | remaining ≈ 0m 22s | indomain_Memo2496/baseline/seed2 → epoch 9, source-val CCC 0.728 (4s)
[16:21:15]   Part B in-domain reference runs: 6/9 (67%) | elapsed 0m 32s | remaining ≈ 0m 16s | indomain_Memo

,domain,n_songs,in_domain_ccc,unseen_baseline_ccc,unseen_domain alignment_ccc,unseen_emotion alignment_ccc,gap,gap_closed_by_domain alignment,gap_closed_by_emotion alignment
0,MediaEval,358,0.7998,0.7004,0.7048,0.6957,0.0994,0.0445,-0.0478
1,Memo2496,483,0.7305,0.5865,0.5933,0.6066,0.1440,0.0475,0.1401
2,PMEmo,151,0.7840,0.6454,0.6252,0.6508,0.1386,-0.1457,0.0389


[16:21:20] ✅ Part B finished.


# Part C — Evidence analyses on source data (no test labels)

1. **Emotion probe:** a simple linear model reads valence/arousal from the representation z (fitted on source-training songs, scored with CCC on source-validation songs). Higher = more emotion information kept.
2. **Domain probe:** a simple linear classifier guesses the domain from z (balanced accuracy on source-validation songs; 50 % = no domain information).
3. **Cross-domain neighbour error:** for each validation song, the 5 most similar songs (in z) *from the other training domain*; average emotion difference in points (0–100). Lower = emotions aligned across domains.
4. **Domain distance (audio only):** how easily a linear classifier separates two domains using the average MERT features (5-fold cross-validated balanced accuracy, and the proxy A-distance 2·(1 − 2·error); 0 = indistinguishable, 2 = completely separable). Uses no ratings.

In [10]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

probe_rows = []
progress = Progress(len(FOLDS) * 3 * len(PROBE_SEEDS), 'Probes', every_s=20)
for t in FOLDS:
    src = FOLDS[t]
    mean, std = stats_of(f'fold_{t}')
    tr = np.concatenate([tracks(d, 'train') for d in src])
    va = np.concatenate([tracks(d, 'val') for d in src])
    dom_tr = catalog.loc[tr, 'domain'].to_numpy()
    dom_va = catalog.loc[va, 'domain'].to_numpy()
    y_tr = catalog.loc[tr, ['valence', 'arousal']].to_numpy()
    y_va = catalog.loc[va, ['valence', 'arousal']].to_numpy()
    for label, m, w, scope in fold_models(t)[:3]:
        for seed in PROBE_SEEDS:
            r = json.loads((RUN_DIR / 'runs' / fold_spec(t, m, w, scope, seed)['name'] / 'result.json').read_text())
            model = load_model(r)
            raw_tr = embed(model, tr, mean, std)
            scaler = StandardScaler().fit(raw_tr)
            z_tr, z_va = scaler.transform(raw_tr), scaler.transform(embed(model, va, mean, std))
            emo = metrics(y_va, np.clip(Ridge(alpha=1.0).fit(z_tr, y_tr).predict(z_va), 0, 1))['macro_ccc']
            dom = balanced_accuracy_score(dom_va, LogisticRegression(max_iter=2000, class_weight='balanced').fit(z_tr, dom_tr).predict(z_va))
            unit = z_va / np.maximum(np.linalg.norm(z_va, axis=1, keepdims=True), 1e-8)
            sim = unit @ unit.T
            err = []
            for i in range(len(va)):
                other = np.flatnonzero(dom_va != dom_va[i])
                nn5 = other[np.argsort(-sim[i, other])[:5]]
                err.append(np.abs(y_va[nn5] - y_va[i]).mean())
            probe_rows.append(dict(test_domain=t, model=label, seed=seed, emotion_probe_ccc=emo,
                                   domain_probe_balanced_accuracy=dom, cross_domain_neighbor_error_points=100 * float(np.mean(err))))
            progress.step()
probes = pd.DataFrame(probe_rows)
probes.to_csv(RUN_DIR / 'probes_per_seed.csv', index=False)
display(probes.groupby(['test_domain', 'model'])[['emotion_probe_ccc', 'domain_probe_balanced_accuracy',
                                                  'cross_domain_neighbor_error_points']].mean().round(4))

# Domain distance from raw MERT features (mean of all layers, averaged over each song's windows)
@torch.no_grad()
def song_vectors(t):
    out = []
    for k in range(0, len(t), 64):
        tt = torch.as_tensor(t[k:k + 64], device=DEVICE)
        idx = WIN[tt, :int(LENGTHS[tt].max())]
        m = (idx >= 0).float()
        x = PACK_LAYERS[idx.clamp(min=0)].float().mean(2)
        out.append(((x * m[..., None]).sum(1) / m.sum(1, keepdim=True)).cpu().numpy())
    return np.concatenate(out)
vectors = {d: song_vectors(np.flatnonzero((catalog.domain == d).to_numpy())) for d in ALL_DOMAINS}
dist_rows = []
for i, a in enumerate(ALL_DOMAINS):
    for b in ALL_DOMAINS[i + 1:]:
        X = StandardScaler().fit_transform(np.concatenate([vectors[a], vectors[b]]))
        lab = np.array([0] * len(vectors[a]) + [1] * len(vectors[b]))
        p = cross_val_predict(LogisticRegression(max_iter=2000, class_weight='balanced', C=0.1), X, lab,
                              cv=StratifiedKFold(5, shuffle=True, random_state=0))
        acc = balanced_accuracy_score(lab, p)
        dist_rows.append(dict(domain_a=a, domain_b=b, balanced_accuracy=acc, proxy_a_distance=2 * (1 - 2 * (1 - acc))))
distances = pd.DataFrame(dist_rows)
distances.to_csv(RUN_DIR / 'domain_distances.csv', index=False)
display(distances.round(4))
log('✅ Part C finished.')

[16:21:22] ▶ Probes: 45 to do
[16:21:22]   Probes: 1/45 (2%) | elapsed 0m 00s | remaining ≈ 0m 11s 
[16:21:33]   Probes: 45/45 (100%) | elapsed 0m 11s | remaining ≈ 0m 00s 


emotion_probe_ccc  \
test_domain model                                  
MediaEval   baseline                      0.7479   
            domain alignment              0.7431   
            emotion alignment             0.7575   
Memo2496    baseline                      0.8222   
            domain alignment              0.8164   
            emotion alignment             0.8316   
PMEmo       baseline                      0.7441   
            domain alignment              0.7433   
            emotion alignment             0.7535   

                               domain_probe_balanced_accuracy  \
test_domain model                                               
MediaEval   baseline                                   0.9620   
            domain alignment                           0.9559   
            emotion alignment                          0.9787   
Memo2496    baseline                                   0.9272   
            domain alignment                           0.9077   
            emotion alignment                          0.9467   
PMEmo       baseline                                   0.7972   
            domain alignment                           0.7742   
            emotion alignment                          0.8068   

                               cross_domain_neighbor_error_points  
test_domain model                                                  
MediaEval   baseline                                      15.1047  
            domain alignment                              14.9271  
            emotion alignment                             13.9253  
Memo2496    baseline                                      13.5633  
            domain alignment                              14.0622  
            emotion alignment                             11.9985  
PMEmo       baseline                                      12.2974  
            domain alignment                              12.2851  
            emotion alignment                             11.8841

,domain_a,domain_b,balanced_accuracy,proxy_a_distance
0,MediaEval,Memo2496,0.8974,1.5894
1,MediaEval,PMEmo,0.9683,1.8733
2,Memo2496,PMEmo,0.9823,1.9291


[16:21:48] ✅ Part C finished.


# Part D — Final independent test on Soundtracks (never used before)

1. **Strengths for the final models** follow the rule declared in Step 1 (most frequent choice across the three folds, otherwise the median); they are saved in `final_lock.json` before Soundtracks is opened.
2. **Final models** are trained on the training songs of **all three domains** (best epoch on the source-validation songs of all three), 10 seeds, for baseline, domain alignment, emotion alignment and the all-pairs control.
3. Only then: the Soundtracks ratings are read from the original archive on Drive, their checksum is compared with the one recorded by the Data Preparation notebook (proving they were not changed), and the models are evaluated once. *Energy* is used as the arousal rating.

In [11]:
final_lock_path = RUN_DIR / 'final_lock.json'
def rule(values):
    values = sorted(values)
    counts = {v: values.count(v) for v in values}
    top = max(counts.values())
    return min(v for v, c in counts.items() if c == top) if top > 1 else values[len(values) // 2]
if final_lock_path.is_file():
    FINAL = json.loads(final_lock_path.read_text())['weights']
else:
    FINAL = {m: rule([SELECTION[t][m] for t in FOLDS]) for m in ('dann', 'emotion')}
    atomic_json(final_lock_path, dict(weights=FINAL, rule=FINAL_WEIGHT_RULE, locked=datetime.datetime.now().isoformat()))
print('Final strengths:', FINAL)

def final_spec(method, weight, scope, seed):
    return dict(name=f'final_all3/{method}_w{weight}_{scope}/seed{seed}', stats_key='final_all3',
                train={d: tracks(d, 'train') for d in ALL_DOMAINS}, val={d: tracks(d, 'val') for d in ALL_DOMAINS},
                method=method, weight=weight, scope=scope, seed=seed)
FINAL_MODELS = [('baseline', 'baseline', 0.0, 'none'), ('domain alignment', 'dann', FINAL['dann'], 'none'),
                ('emotion alignment', 'emotion', FINAL['emotion'], 'cross'),
                ('control: emotion all-pairs', 'emotion', FINAL['emotion'], 'all')]
final_runs = pd.DataFrame(run_all([final_spec(m, w, s, seed) for _, m, w, s in FINAL_MODELS for seed in SEEDS],
                                  'Part D final training runs (all three domains)'))
atomic_json(RUN_DIR / 'final_models_sealed.json', dict(runs=final_runs['name'].tolist(), sealed=datetime.datetime.now().isoformat()))

# ---- Only now: open the Soundtracks ratings and verify they are unchanged ----
lock = json.loads((DATA_DIR / 'catalog' / 'soundtracks_lock.json').read_text())
target_name = Path(lock['ratings_file']).name
ratings_bytes = None
for path in sorted((DATASETS_DIR / 'Soundtracks').rglob('*')):
    if path.is_file() and path.name == target_name:
        ratings_bytes = path.read_bytes()
    elif path.is_file() and path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as z:
            for info in z.infolist():
                if Path(info.filename).name == target_name and '__MACOSX' not in info.filename:
                    ratings_bytes = z.read(info)
    if ratings_bytes is not None:
        break
if ratings_bytes is None:
    raise FileNotFoundError(f'{target_name} not found under {DATASETS_DIR / "Soundtracks"}')
if hashlib.sha256(ratings_bytes).hexdigest() != lock['ratings_sha256']:
    raise RuntimeError('Soundtracks ratings differ from the file recorded by Notebook 16. Stop and check.')
log('Soundtracks ratings checksum verified (identical to Notebook 16).')
ratings = pd.read_csv(io.BytesIO(ratings_bytes))
ratings.columns = [c.strip() for c in ratings.columns]
ratings['track_id'] = ratings['Number'].astype(int).astype(str).str.zfill(3)
st = catalog_all[catalog_all.dataset == 'Soundtracks'].merge(ratings[['track_id', 'Valence', 'Energy']], on='track_id', how='left')
if st[['Valence', 'Energy']].isna().any().any():
    raise ValueError('Some Soundtracks recordings have no rating row.')
st_truth = (st[['Valence', 'Energy']].to_numpy(dtype=float) - 1) / 8
mf = manifest_all.set_index('uid')

@torch.no_grad()
def predict_soundtracks(model, mean, std):
    model.eval()
    preds = []
    for uid in st['uid']:
        with np.load(DATA_DIR / mf.loc[uid, 'feature_relpath'], allow_pickle=False) as f:
            x = torch.from_numpy(f['layers'].astype(np.float32)).to(DEVICE)[None]
        mask = torch.ones(1, x.shape[1], dtype=torch.bool, device=DEVICE)
        preds.append(model((x - mean) / std, mask)[0].clamp(0, 1).cpu().numpy()[0])
    return np.stack(preds)

mean, std = stats_of('final_all3')
st_rows, st_pred_rows = [], []
progress = Progress(len(FINAL_MODELS) * len(SEEDS), 'Soundtracks evaluation', every_s=20)
for label, m, w, scope in FINAL_MODELS:
    for seed in SEEDS:
        r = json.loads((RUN_DIR / 'runs' / final_spec(m, w, scope, seed)['name'] / 'result.json').read_text())
        pred = predict_soundtracks(load_model(r), mean, std)
        st_rows.append(dict(test_domain='Soundtracks', model=label, weight=w, seed=seed, **metrics(st_truth, pred)))
        st_pred_rows.append(pd.DataFrame(dict(test_domain='Soundtracks', model=label, seed=seed, uid=st['uid'].values,
                                              true_valence=st_truth[:, 0], pred_valence=pred[:, 0],
                                              true_arousal=st_truth[:, 1], pred_arousal=pred[:, 1])))
        progress.step()
st_results = pd.DataFrame(st_rows)
st_predictions = pd.concat(st_pred_rows)
st_results.to_csv(RUN_DIR / 'soundtracks_results_per_seed.csv', index=False)
st_predictions.to_csv(RUN_DIR / 'soundtracks_predictions.csv', index=False)
log('✅ Part D finished.')

Final strengths: {'dann': 0.03, 'emotion': 0.2}
[16:21:49] ▶ Part D final training runs (all three domains): 40 to do
[16:22:13]   Part D final training runs (all three domains): 1/40 (2%) | elapsed 0m 24s | remaining ≈ 16m 04s | final_all3/baseline_w0.0_none/seed1 → epoch 11, source-val CCC 0.780 (25s)
[16:22:38]   Part D final training runs (all three domains): 2/40 (5%) | elapsed 0m 49s | remaining ≈ 15m 37s | final_all3/baseline_w0.0_none/seed2 → epoch 9, source-val CCC 0.781 (25s)
[16:23:02]   Part D final training runs (all three domains): 3/40 (8%) | elapsed 1m 13s | remaining ≈ 15m 09s | final_all3/baseline_w0.0_none/seed3 → epoch 9, source-val CCC 0.781 (24s)
[16:23:27]   Part D final training runs (all three domains): 4/40 (10%) | elapsed 1m 38s | remaining ≈ 14m 44s | final_all3/baseline_w0.0_none/seed4 → epoch 6, source-val CCC 0.774 (25s)
[16:23:51]   Part D final training runs (all three domains): 5/40 (12%) | elapsed 2m 02s | remaining ≈ 14m 20s | final_all3/baseline_w0.

# Part E — Results

For every test set: mean ± SD over the 10 seeds (CCC primary; MAE/RMSE ×100 = points). Paired comparisons use the same seeds; *better* = at least 8 of 10 seeds better **and** a 95 % bootstrap interval (test songs resampled, seed-averaged predictions) above zero. For Soundtracks, "arousal" means the *energy* rating.

In [12]:
all_results = pd.concat([fold_results, st_results], ignore_index=True)
all_predictions = pd.concat([fold_predictions.drop(columns='split'), st_predictions], ignore_index=True)
TEST_SETS = list(FOLDS) + ['Soundtracks']

summary_rows = []
for (t, m), g in all_results.groupby(['test_domain', 'model']):
    row = dict(test_domain=t, model=m, seeds=len(g))
    for k in ('macro_ccc', 'valence_ccc', 'arousal_ccc', 'macro_pearson', 'macro_mae', 'macro_rmse'):
        row[k] = f'{g[k].mean():.3f} ± {g[k].std(ddof=1):.3f}'
        row[k + '_mean'] = g[k].mean()
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN_DIR / 'summary.csv', index=False)
for t in TEST_SETS:
    title = 'FINAL INDEPENDENT TEST' if t == 'Soundtracks' else f'trained on {" + ".join(FOLDS[t])}'
    print(f'\n━━━ Test: {t} ({title}) ━━━')
    display(summary[summary.test_domain == t].set_index('model')[['seeds', 'macro_ccc', 'valence_ccc', 'arousal_ccc',
                                                                  'macro_pearson', 'macro_mae', 'macro_rmse']])

def paired(t, a, b='baseline'):
    ra = all_results[(all_results.test_domain == t) & (all_results.model == a)].set_index('seed')['macro_ccc']
    rb = all_results[(all_results.test_domain == t) & (all_results.model == b)].set_index('seed')['macro_ccc']
    diffs = (ra - rb).dropna()
    pa = all_predictions[(all_predictions.test_domain == t) & (all_predictions.model == a)]
    pb = all_predictions[(all_predictions.test_domain == t) & (all_predictions.model == b)]
    cols = ['pred_valence', 'pred_arousal']
    ma, mb = pa.groupby('uid')[cols].mean(), pb.groupby('uid')[cols].mean()
    truth = pa.groupby('uid')[['true_valence', 'true_arousal']].first().loc[ma.index].to_numpy().T
    pred_a, pred_b = ma.to_numpy().T, mb.loc[ma.index].to_numpy().T
    rng = np.random.default_rng(zlib.crc32(f'{t}|{a}|{b}'.encode()))
    idx = rng.integers(0, truth.shape[1], size=(BOOTSTRAP_REPS, truth.shape[1]))
    boot = (ccc_batch(truth[:, idx], pred_a[:, idx]) - ccc_batch(truth[:, idx], pred_b[:, idx])).mean(0)
    low, high = np.percentile(boot, [2.5, 97.5])
    wins = int((diffs > 0).sum())
    return dict(test_domain=t, comparison=f'{a} − {b}', mean_seed_difference=diffs.mean(), seeds_better=f'{wins}/{len(diffs)}',
                ci95_low=low, ci95_high=high, better=bool(wins >= BETTER_RULE['min_seed_wins'] and low > 0),
                worse=bool(len(diffs) - wins >= BETTER_RULE['min_seed_wins'] and high < 0))
paired_rows = []
for t in TEST_SETS:
    for a in ('domain alignment', 'emotion alignment', 'control: emotion all-pairs'):
        paired_rows.append(paired(t, a))
    paired_rows.append(paired(t, 'emotion alignment', 'domain alignment'))
    paired_rows.append(paired(t, 'emotion alignment', 'control: emotion all-pairs'))
paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(RUN_DIR / 'paired_comparisons.csv', index=False)
print('\nPaired comparisons (macro CCC, positive = first model better):')
display(paired_df.round(4))

mean_ccc = all_results.groupby(['test_domain', 'model'])['macro_ccc'].mean().unstack()
criterion = {t: bool(mean_ccc.loc[t, 'emotion alignment'] > mean_ccc.loc[t, 'domain alignment'] > mean_ccc.loc[t, 'baseline'])
             for t in TEST_SETS}
print('\nRanking emotion > domain > baseline (mean macro CCC):', {t: '✓' if v else '✗' for t, v in criterion.items()})
layer_w = np.array([r for r in partA[partA.method == 'baseline']['layer_weights']]).mean(0)
print('\nMean learned layer weights (baseline, all folds; 0 = input embedding, 12 = last layer):')
print(pd.Series(layer_w, index=range(N_LAYERS)).round(3).to_string())
atomic_json(RUN_DIR / 'criterion.json', dict(criterion=criterion, mean_macro_ccc=mean_ccc.to_dict(), layer_weights=layer_w.tolist()))


━━━ Test: MediaEval (trained on PMEmo + Memo2496) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,10,0.662 ± 0.016,0.536 ± 0.029,0.789 ± 0.008,0.682 ± 0.014,0.093 ± 0.002,0.117 ± 0.003
control: emotion all-pairs,10,0.662 ± 0.019,0.535 ± 0.032,0.789 ± 0.014,0.687 ± 0.015,0.092 ± 0.003,0.116 ± 0.003
domain alignment,10,0.661 ± 0.013,0.542 ± 0.023,0.780 ± 0.010,0.689 ± 0.014,0.094 ± 0.003,0.119 ± 0.003
emotion alignment,10,0.655 ± 0.020,0.519 ± 0.035,0.790 ± 0.013,0.679 ± 0.021,0.091 ± 0.002,0.115 ± 0.003



━━━ Test: Memo2496 (trained on MediaEval + PMEmo) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,10,0.559 ± 0.013,0.473 ± 0.015,0.645 ± 0.016,0.663 ± 0.007,0.139 ± 0.005,0.174 ± 0.006
control: emotion all-pairs,10,0.589 ± 0.012,0.504 ± 0.017,0.673 ± 0.016,0.667 ± 0.006,0.129 ± 0.006,0.159 ± 0.006
domain alignment,10,0.567 ± 0.018,0.480 ± 0.021,0.654 ± 0.021,0.657 ± 0.007,0.137 ± 0.007,0.170 ± 0.008
emotion alignment,10,0.583 ± 0.015,0.497 ± 0.016,0.669 ± 0.022,0.667 ± 0.003,0.131 ± 0.006,0.161 ± 0.008



━━━ Test: PMEmo (trained on MediaEval + Memo2496) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,10,0.604 ± 0.022,0.523 ± 0.049,0.685 ± 0.012,0.698 ± 0.022,0.106 ± 0.004,0.129 ± 0.004
control: emotion all-pairs,10,0.609 ± 0.015,0.525 ± 0.021,0.693 ± 0.019,0.703 ± 0.011,0.105 ± 0.003,0.128 ± 0.003
domain alignment,10,0.586 ± 0.026,0.495 ± 0.038,0.678 ± 0.017,0.693 ± 0.018,0.108 ± 0.004,0.131 ± 0.004
emotion alignment,10,0.609 ± 0.018,0.524 ± 0.020,0.695 ± 0.024,0.709 ± 0.015,0.105 ± 0.003,0.128 ± 0.003



━━━ Test: Soundtracks (FINAL INDEPENDENT TEST) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,10,0.385 ± 0.021,0.356 ± 0.035,0.414 ± 0.024,0.648 ± 0.014,0.229 ± 0.013,0.272 ± 0.012
control: emotion all-pairs,10,0.428 ± 0.025,0.404 ± 0.036,0.452 ± 0.029,0.671 ± 0.017,0.213 ± 0.016,0.254 ± 0.015
domain alignment,10,0.411 ± 0.026,0.378 ± 0.033,0.444 ± 0.024,0.655 ± 0.023,0.221 ± 0.013,0.264 ± 0.014
emotion alignment,10,0.431 ± 0.025,0.407 ± 0.038,0.455 ± 0.027,0.671 ± 0.019,0.213 ± 0.017,0.254 ± 0.016



Paired comparisons (macro CCC, positive = first model better):


,test_domain,comparison,mean_seed_difference,seeds_better,ci95_low,ci95_high,better,worse
0,MediaEval,domain alignment − baseline,-0.0012,3/10,-0.0081,0.0010,False,False
1,MediaEval,emotion alignment − baseline,-0.0076,2/10,-0.0204,-0.0091,False,True
2,MediaEval,control: emotion all-pairs − baseline,-0.0003,4/10,-0.0109,-0.0005,False,False
3,MediaEval,emotion alignment − domain alignment,-0.0063,2/10,-0.0180,-0.0042,False,True
4,MediaEval,emotion alignment − control: emotion all-pairs,-0.0072,4/10,-0.0110,-0.0064,False,False
5,Memo2496,domain alignment − baseline,0.0082,6/10,0.0074,0.0110,False,False
6,Memo2496,emotion alignment − baseline,0.0243,10/10,0.0194,0.0255,True,False
7,Memo2496,control: emotion all-pairs − baseline,0.0298,9/10,0.0248,0.0312,True,False
8,Memo2496,emotion alignment − domain alignment,0.0162,8/10,0.0103,0.0164,True,False
9,Memo2496,emotion alignment − control: emotion all-pairs,-0.0054,4/10,-0.0067,-0.0045,False,False



Ranking emotion > domain > baseline (mean macro CCC): {'MediaEval': '✗', 'Memo2496': '✓', 'PMEmo': '✗', 'Soundtracks': '✓'}

Mean learned layer weights (baseline, all folds; 0 = input embedding, 12 = last layer):
0     0.076
1     0.062
2     0.066
3     0.059
4     0.081
5     0.104
6     0.114
7     0.094
8     0.082
9     0.067
10    0.072
11    0.066
12    0.058


## Step F — Review file

Packs protocol, locks, all run results and histories, test results, predictions, reference, probes, distances, summary and paired tests into `final_review.zip` (no checkpoints).

In [13]:
review = RUN_DIR / 'final_review.zip'
with zipfile.ZipFile(review, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path != review and path.suffix in ('.json', '.csv', '.gz'):
            z.write(path, path.relative_to(RUN_DIR).as_posix())
log(f'✅ Review file ready on Drive: {review}')
try:
    from google.colab import files
    files.download(str(review))
except Exception as error:
    print('Automatic download not available; take the file from Drive instead.', error)

[16:41:20] ✅ Review file ready on Drive: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/experiments/nb18_97de095f25de/final_review.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>